In [1]:
%pip install mlflow fastapi uvicorn scikit-learn pandas numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

PROJECT_DIR = r"D:\kesava 2\231FA04B56"
DB_PATH = "sqlite:///D:/kesava 2/231FA04B56/mlflow.db"

mlflow.set_tracking_uri(DB_PATH)

print("MLflow:", mlflow.get_tracking_uri())

MLflow: sqlite:///D:/kesava 2/231FA04B56/mlflow.db


In [3]:
# Load dataset
DATA_PATH = os.path.join(PROJECT_DIR, "bank_churn.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.head())

# Target
target = "churn"

# Remove ID
X = df.drop(columns=["customer_id", "churn"])
y = df["churn"]

# Categorical and numerical columns
categorical_features = ["geography", "gender", "has_cr_card", "is_active_member"]

numerical_features = [
    "credit_score",
    "age",
    "tenure_years",
    "balance",
    "num_products",
    "estimated_salary",
    "monthly_transactions",
    "complaints_last_year"
]

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=4,
    random_state=42
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

# Evaluation
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print("\nModel Results")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC-AUC  :", roc_auc)

Dataset shape: (6120, 14)
   customer_id  credit_score geography  gender        age  tenure_years  \
0       102094    743.520061   Germany  Female  38.460855             1   
1       101612    729.479880   Germany  FEMALE  29.115844             7   
2       105858           NaN   Germany    Male  33.687474             0   
3       104294    758.594734     Spain    Male  39.134293            10   
4       102571    519.626773    France  FEMALE  42.524934             4   

        balance  num_products has_cr_card is_active_member  estimated_salary  \
0  83722.026007             2          No               No      29157.083621   
1  68053.038287             1          No              Yes     121774.883439   
2  92560.217237             1          No               No     121329.645742   
3   8811.863054             1         Yes               No     150230.538972   
4  24966.092387             3         Yes              Yes      47438.911368   

   monthly_transactions  complaints_last_y

In [6]:
# Create/use a NEW experiment for Lab 4
EXPERIMENT_NAME = "Bank_Churn_Lab4"

client = mlflow.MlflowClient()

experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    experiment_id = client.create_experiment(
        EXPERIMENT_NAME,
        artifact_location="file:///D:/kesava 2/231FA04B56/mlruns/lab4"
    )
else:
    experiment_id = experiment.experiment_id

print("Experiment ID:", experiment_id)

# Train/log model in MLflow
with mlflow.start_run(
    experiment_id=experiment_id,
    run_name="Bank_Churn_RandomForest_Lab4"
) as run:

    mlflow.log_param("model", "Random Forest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 15)
    mlflow.log_param("min_samples_split", 4)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)

    mlflow.sklearn.log_model(
        pipeline,
        name="model",
        skops_trusted_types=["numpy.dtype"]
    )

    run_id = run.info.run_id

model_uri = f"runs:/{run_id}/model"

print("\nRun ID:", run_id)
print("Model URI:", model_uri)

# Register model
MODEL_NAME = "BankChurnLab4Model"

registered = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME
)

print("Registered Version:", registered.version)

# Set champion alias
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=str(registered.version)
)

print("Champion Version:", registered.version)

# Test loading
loaded_model = mlflow.sklearn.load_model(
    f"models:/{MODEL_NAME}@champion"
)

print("\nMODEL LOADED SUCCESSFULLY!")

Experiment ID: 2


Successfully registered model 'BankChurnLab4Model'.
2026/09/18 14:09:18 WARNING mlflow.tracking._model_registry.fluent: Run with id 7013f43099ef4820a0f12c0e901e6020 has no artifacts at artifact path 'model', registering model based on models:/m-6f6316f2f71748b4bd00bdf0149e2580 instead



Run ID: 7013f43099ef4820a0f12c0e901e6020
Model URI: runs:/7013f43099ef4820a0f12c0e901e6020/model
Registered Version: 1
Champion Version: 1


Created version '1' of model 'BankChurnLab4Model'.



MODEL LOADED SUCCESSFULLY!


In [7]:
%%writefile app.py

from typing import Any

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from fastapi import FastAPI, HTTPException

# --------------------------------------------------
# MLflow configuration
# --------------------------------------------------

mlflow.set_tracking_uri(
    "sqlite:///D:/kesava 2/231FA04B56/mlflow.db"
)

MODEL_NAME = "BankChurnLab4Model"
MODEL_URI = f"models:/{MODEL_NAME}@champion"

# --------------------------------------------------
# Load model
# --------------------------------------------------

try:
    model = mlflow.sklearn.load_model(MODEL_URI)
    model_loaded = True

    print("===================================")
    print("MODEL LOADED SUCCESSFULLY")
    print("Model:", MODEL_NAME)
    print("URI:", MODEL_URI)
    print("===================================")

except Exception as e:
    model = None
    model_loaded = False
    print("MODEL LOADING FAILED:", e)

# --------------------------------------------------
# FastAPI
# --------------------------------------------------

app = FastAPI(
    title="Bank Churn Prediction API",
    description="Bank Customer Churn Prediction using MLflow",
    version="1.0"
)

# --------------------------------------------------
# Home
# --------------------------------------------------

@app.get("/")
def home():
    return {
        "message": "Bank Churn Prediction API is running",
        "model": MODEL_NAME,
        "alias": "champion"
    }

# --------------------------------------------------
# Health
# --------------------------------------------------

@app.get("/health")
def health():

    if model_loaded:
        return {
            "status": "healthy",
            "model_loaded": True,
            "model": MODEL_NAME,
            "model_uri": MODEL_URI
        }

    raise HTTPException(
        status_code=503,
        detail="Model is not loaded"
    )

# --------------------------------------------------
# Features
# --------------------------------------------------

@app.get("/features")
def features():

    return {
        "required_features": [
            "credit_score",
            "geography",
            "gender",
            "age",
            "tenure_years",
            "balance",
            "num_products",
            "has_cr_card",
            "is_active_member",
            "estimated_salary",
            "monthly_transactions",
            "complaints_last_year"
        ]
    }

# --------------------------------------------------
# Prediction
# --------------------------------------------------

@app.post("/predict")
def predict(data: dict[str, Any]):

    if not model_loaded:
        raise HTTPException(
            status_code=503,
            detail="Model is not loaded"
        )

    required_features = [
        "credit_score",
        "geography",
        "gender",
        "age",
        "tenure_years",
        "balance",
        "num_products",
        "has_cr_card",
        "is_active_member",
        "estimated_salary",
        "monthly_transactions",
        "complaints_last_year"
    ]

    missing = [
        feature
        for feature in required_features
        if feature not in data
    ]

    if missing:
        raise HTTPException(
            status_code=400,
            detail={
                "message": "Missing required features",
                "missing_features": missing
            }
        )

    try:

        input_df = pd.DataFrame([data])

        input_df = input_df[required_features]

        prediction = model.predict(input_df)[0]

        result = {
            "prediction": int(prediction)
        }

        if hasattr(model, "predict_proba"):

            probability = model.predict_proba(input_df)[0]

            result["probability_no_churn"] = float(probability[0])
            result["probability_churn"] = float(probability[1])

        return result

    except Exception as e:

        raise HTTPException(
            status_code=500,
            detail=f"Prediction failed: {str(e)}"
        )

Overwriting app.py
